# 26: Does Candidate Subsampling Change Causal Amplification?

This notebook preserves the experiment history rather than converting it into a clean success/failure story. The executable question is whether changing evaluation breadth changes aggregate downstream causal amplification once expected background construction is dynamically matched.

In [1]:
from pathlib import Path
import json
import math
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_PROFILE = os.environ.get("NOTEBOOK_PROFILE", "quick")
RUN_CANONICAL = os.environ.get("RUN_CANONICAL", "0") == "1"


def find_repo_root(start=Path.cwd()):
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "content" / "books" / "digital-life").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate repository root")

REPO_ROOT = find_repo_root()
NOTEBOOK_DIR = REPO_ROOT / "notebooks"
FIG_DIR = NOTEBOOK_DIR / "generated-figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def load_json(relative_path):
    path = REPO_ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def require_path(relative_path):
    path = REPO_ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def summarize_result(label, result, status=None, source="canonical research artifact"):
    row = {"label": label, "source": source}
    if status is not None:
        row["status"] = status
    for key in ["n", "mean", "ci95_low", "ci95_high", "achieved_mde80_one_sided"]:
        if key in result:
            row[key] = result[key]
    return row

print("profile", NOTEBOOK_PROFILE, "run_canonical", RUN_CANONICAL)
print("repo", REPO_ROOT)

CHAPTER = 26
MANUSCRIPT = require_path("content/books/digital-life/26-does-candidate-subsampling-change-causal-amplification/index.md")
LINEAGE = [
    "scripts/books/digital-life/ch26_digital_crystal_matched_rate_causal_amplification_v1.py",
    "scripts/books/digital-life/ch26_v1_audit_true_unbounded_and_rate_drift.py",
    "scripts/books/digital-life/ch26_digital_crystal_dynamically_matched_rate_causal_amplification_v2.py",
    "scripts/books/digital-life/ch26_v2_analytic_audit.py",
    "scripts/books/digital-life/ch26_v2_zero_inflation_two_channel_audit.py",
]
for item in LINEAGE:
    require_path(item)
print("manuscript", MANUSCRIPT.relative_to(REPO_ROOT))
print("lineage ok", len(LINEAGE))

profile quick run_canonical False
repo C:\Projects\working-book
manuscript content\books\digital-life\26-does-candidate-subsampling-change-causal-amplification\index.md
lineage ok 5


## Claim Boundary

V1 is not negative evidence against causal amplification. Its reference construction was inadequate because calibration matched an early state rather than the changing dynamical process. V2 asks the narrower question: under dynamically matched expected background construction, does strong finite subsampling amplify downstream causal consequence relative to true unbounded evaluation at the frozen ±0.15 attachment scale?

In [2]:
v1_verdict = load_json("research/digital-life/ch26-matched-rate-causal-amplification-v1/stage-06-verdict.json")
v1_audit = load_json("research/digital-life/ch26-v1-audit/ch26-v1-audit-report.json")
v2_primary = load_json("research/digital-life/ch26-dynamically-matched-rate-causal-amplification-v2/stage-04-primary-test.json")
v2_matching = load_json("research/digital-life/ch26-dynamically-matched-rate-causal-amplification-v2/stage-06-per-lag-matching.json")
v2_verdict = load_json("research/digital-life/ch26-dynamically-matched-rate-causal-amplification-v2/stage-07-verdict.json")
mechanism = load_json("research/digital-life/ch26-v2-mechanism-audit/ch26-v2-mechanism-audit-report.json")

pd.DataFrame([
    {"version": "V1", "role": "historical construct", "status": v1_audit.get("overall_status", v1_verdict.get("overall_status", "AUDITED")), "source": "canonical research artifact"},
    {"version": "V2", "role": "confirmatory primary", "status": v2_primary["status"], "source": "canonical research artifact"},
])

,version,role,status,source
0,V1,historical construct,CAUSAL_AMPLIFICATION_INVARIANT_WITHIN_SEI,canonical research artifact
1,V2,confirmatory primary,BOUNDED_NEAR_ZERO,canonical research artifact


## Quick Demonstration: Dynamic Matching Is a Process Constraint

This toy calculation is recomputed. It is not the Digital Crystal estimator. It shows why a one-time calibration can drift when the background process changes with lag, while dynamic matching keeps each lag aligned.

In [3]:
lags = list(range(1, 13))
fixed_reference = [40.0 for _ in lags]
changing_prevent_target = [37.0 + 0.42 * lag + 0.9 * math.sin(lag / 2.5) for lag in lags]
dynamic_reference = changing_prevent_target[:]
quick_matching = pd.DataFrame({
    "lag": lags,
    "prevent_target": changing_prevent_target,
    "fixed_reference_error": [a - b for a, b in zip(fixed_reference, changing_prevent_target)],
    "dynamic_reference_error": [a - b for a, b in zip(dynamic_reference, changing_prevent_target)],
})
quick_matching

,lag,prevent_target,fixed_reference_error,dynamic_reference_error
0,1,37.770477,2.229523,0.0
1,2,38.485620,1.514380,0.0
2,3,39.098835,0.901165,0.0
3,4,39.579616,0.420384,0.0
4,5,39.918368,0.081632,0.0
5,6,40.127917,-0.127917,0.0
6,7,40.241489,-0.241489,0.0
7,8,40.307463,-0.307463,0.0
8,9,40.381732,-0.381732,0.0
9,10,40.518878,-0.518878,0.0


In [4]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.axhline(0, color="#333333", lw=1)
ax.plot(quick_matching["lag"], quick_matching["fixed_reference_error"], marker="o", label="fixed early calibration")
ax.plot(quick_matching["lag"], quick_matching["dynamic_reference_error"], marker="o", label="dynamic matching")
ax.set_title("ASSERTION / DEMO: dynamic matching prevents lag drift")
ax.set_xlabel("lag")
ax.set_ylabel("construction error vs target")
ax.legend(frameon=False)
path = FIG_DIR / "ch26-dynamic-matching-demo.png"
fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)
path.relative_to(REPO_ROOT)

WindowsPath('notebooks/generated-figures/ch26-dynamic-matching-demo.png')

## Canonical Artifact Audit

The next table is loaded from committed research artifacts. It is not recomputed in this notebook run.

In [5]:
primary_rows = [summarize_result(v2_primary["contrast"], v2_primary["result"], v2_primary["status"])]
pd.DataFrame(primary_rows)

,label,source,status,n,mean,ci95_low,ci95_high,achieved_mde80_one_sided
0,G_T(f=0.10) - G_T(unbounded),canonical research artifact,BOUNDED_NEAR_ZERO,192,0.001302,-0.089844,0.088542,0.115361


In [6]:
rows = []
for lag, arms in v2_matching.items():
    for arm, values in arms.items():
        if not isinstance(values, dict) or "target" not in values:
            continue
        target = values["target"]["mean"]
        prevent = values["prevent_expected"]["mean"]
        force = values["force_expected"]["mean"]
        rows.append({"lag": int(lag), "arm": arm, "prevent_minus_target": prevent - target, "force_minus_target": force - target})
match_df = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(7.4, 3.4))
for arm, group in match_df.groupby("arm"):
    ax.plot(group["lag"], group["prevent_minus_target"], marker="o", label=f"{arm} PREVENT-target")
ax.axhline(0, color="#333333", lw=1)
ax.set_title("CANONICAL ARTIFACT: dynamic PREVENT target matching")
ax.set_xlabel("lag")
ax.set_ylabel("mean expected construction error")
ax.legend(frameon=False, fontsize=8)
path = FIG_DIR / "ch26-canonical-target-matching.png"
fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)
print(path.relative_to(REPO_ROOT))
match_df.groupby("arm")[["prevent_minus_target", "force_minus_target"]].agg(["mean", "max", "min"])

notebooks\generated-figures\ch26-canonical-target-matching.png


prevent_minus_target                             force_minus_target  \
                          mean           max           min               mean   
arm                                                                             
f=0.10            0.000000e+00  0.000000e+00  0.000000e+00           0.002913   
f=0.25            6.329752e-13  7.531753e-12 -5.769607e-12           0.004181   
f=0.50           -7.342275e-14  5.975664e-12 -4.568790e-12           0.004359   
f=0.75           -3.807325e-13  3.282707e-12 -7.112533e-12           0.001878   
f=1.00           -7.217930e-13  5.677236e-12 -4.817480e-12          -0.003109   
unbounded        -7.217930e-13  5.677236e-12 -4.817480e-12           0.012096   

                               
                max       min  
arm                            
f=0.10     0.041399 -0.007343  
f=0.25     0.045384 -0.007651  
f=0.50     0.048276 -0.007447  
f=0.75     0.049333 -0.007529  
f=1.00     0.035732 -0.011346  
unbounded  0.092591  0.003228

In [7]:
channel = mechanism["audit_B_exact_ring1_channels"]
channel_rows = []
for arm, values in channel.get("by_arm", {}).items():
    flat = {"arm": arm, "source": "canonical research artifact"}
    for name, stat in values.items():
        if isinstance(stat, dict) and "mean" in stat:
            flat[name] = stat["mean"]
    channel_rows.append(flat)
channel_df = pd.DataFrame(channel_rows)
channel_df

""


In [8]:
if not channel_df.empty:
    plot_cols = [c for c in channel_df.columns if c not in {"arm", "source"}][:6]
    ax = channel_df.set_index("arm")[plot_cols].plot(kind="bar", figsize=(8, 3.6))
    ax.set_title("CANONICAL ARTIFACT: pathway routing changes with evaluation breadth")
    ax.set_ylabel("mean channel contribution")
    ax.legend(frameon=False, fontsize=7)
    path = FIG_DIR / "ch26-pathway-routing.png"
    ax.figure.tight_layout(); ax.figure.savefig(path, dpi=160); plt.close(ax.figure)
    print(path.relative_to(REPO_ROOT))

## Result Ledger

- **Final status:** `BOUNDED_NEAR_ZERO` for strong finite subsampling minus unbounded evaluation at the frozen ±0.15 scale.
- **Surviving mechanism:** evaluation breadth changes causal routing, especially between force-only / promotion opportunity and shared probability-shift pathways.
- **Not justified:** “candidate subsampling has no effect” or “routing change proves amplification.”

`CAUSAL ROUTING != CAUSAL AMPLIFICATION`.

In [9]:
ledger = pd.DataFrame([
    {"claim": "V1 primary amplification contrast", "status": "INVALID_REFERENCE", "source": "canonical artifact + audit", "boundary": "not negative evidence"},
    {"claim": "V2 aggregate amplification at ±0.15", "status": v2_primary["status"], "source": "canonical research artifact", "boundary": "bounded near zero, not universal no-effect"},
    {"claim": "mechanistic rerouting", "status": "SUPPORTED / ANALYSIS_ONLY", "source": "canonical mechanism audit", "boundary": "route change is not amplification"},
])
ledger

,claim,status,source,boundary
0,V1 primary amplification contrast,INVALID_REFERENCE,canonical artifact + audit,not negative evidence
1,V2 aggregate amplification at ±0.15,BOUNDED_NEAR_ZERO,canonical research artifact,"bounded near zero, not universal no-effect"
2,mechanistic rerouting,SUPPORTED / ANALYSIS_ONLY,canonical mechanism audit,route change is not amplification
